# Evaluating Domain Adaptation in Financial Sentiment Analysis
## Hyperparameter Tuning: bert_base vs finbert_mlm

**Authors:** Qiyao Zheng, Benjamin Donick, Necati Furkan Colak  
**Repository:** [https://github.com/nfcolak/essential_text](https://github.com/nfcolak/essential_text)  

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nfcolak/essential_text/blob/qiyao/hyperparameter-tuning/notebooks/tuning_hyperparameters_colab.ipynb)

---
### Experiment
This notebook reproduces the learning-rate and batch-size study for two encoders on the locked `sentences_allagree` split of Financial PhraseBank. Setup follows `notebooks/step1_data_and_baselines_colab.ipynb`: clone the repository, install `requirements.txt`, rebuild the locked PhraseBank and SEntFiN splits, audit leakage, and check every file listed in `data/splits/SHA256SUMS`.

Training is delegated to `scripts/run_tuning.py`. The notebook does not override the search space, the seed list, or the decision to score validation only during selection. Those choices stay in the script.

- **Models.** `bert_base` is `google-bert/bert-base-uncased`. `finbert_mlm` is the domain-pretrained checkpoint `yiyanghkust/finbert-pretrain`. The sentiment-finetuned `ProsusAI/finbert` checkpoint is not used.
- **Stage 1, validation-only selection.** Learning rates `2e-5` and `3e-5` are crossed with batch sizes `16` and `32` (eight runs). Each run uses seed `42`. The selection score is validation Macro-F1. The PhraseBank test split and SEntFiN are not loaded in this stage. Ties prefer the lower learning rate, then the smaller batch size. Each model is selected independently.
- **Stage 2, final evaluation.** The selected configuration of each model is trained again with seeds `42`, `100`, and `2026`. These six runs score the locked PhraseBank test set and the external SEntFiN holdout. Seed 42 is retrained here because the selection run does not keep a checkpoint.
- **Outputs.** Tables and run directories are written to `results/tuning/`. The runner does not write into `results/step2`.

### Tokenizer integrity
Before a FinBERT run is trained or reused, the tokenizer must have vocabulary size 30,873, and the terms profit, ebitda, and depreciation must not all encode as `[UNK]`. A completed FinBERT directory is eligible for resume only when `tokenizer_check.json` records that check. Directories that do not are moved to `archive/invalid_finbert_tokenizer/<timestamp>/` and are excluded from selection, from the summary tables, and from the figures. The archived files are kept.

### Runtime, backup, and resumption
Use a GPU runtime (**Runtime → Change runtime type → GPU**) before the training cell. The study is 8 selection runs plus 6 final runs. A completed run is any directory whose `result.json` has status `completed`, subject to the FinBERT receipt above.

Completed runs, tokenizer receipts, and tables are copied to `MyDrive/essential_text_tuning`. After a disconnect, start a new GPU runtime and rerun from the top. The Drive copy is checked before the local tree, and both are checked before restore, so a directory that failed the tokenizer receipt is not copied back into `selection/` or `final/`. Partial `.tmp` directories are not backed up.

The last section only reads completed tables and writes figures. It does not train.


In [ ]:
# Cell 1: Clone or update the repository at the fixed Colab path.
# Rerunning this cell stays on /content/essential_text and does not nest another checkout.
import os
import subprocess
import sys

REPO_URL = "https://github.com/nfcolak/essential_text.git"
REPO_DIR = "/content/essential_text"
BRANCH = "qiyao/hyperparameter-tuning"
GIT_DIR = os.path.join(REPO_DIR, ".git")
TUNING_SCRIPT = os.path.join(REPO_DIR, "scripts", "run_tuning.py")

def _run_git(args: list[str]) -> str:
    completed = subprocess.run(args, check=False, text=True, capture_output=True)
    if completed.returncode != 0:
        detail = (completed.stderr or completed.stdout or "").strip()
        raise RuntimeError(
            "Git command failed (exit {code}): {cmd}\n{detail}".format(
                code=completed.returncode,
                cmd=" ".join(args),
                detail=detail,
            )
        )
    return completed.stdout.strip()

os.makedirs(os.path.dirname(REPO_DIR), exist_ok=True)
if os.path.isdir(GIT_DIR):
    _run_git(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH])
    _run_git(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, "origin/" + BRANCH])
    _run_git(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH])
elif os.path.exists(REPO_DIR):
    raise RuntimeError(
        f"{REPO_DIR} exists but {GIT_DIR} is missing. "
        "Remove that directory and rerun this cell so the clone uses the absolute path."
    )
else:
    _run_git(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR])

os.chdir(REPO_DIR)
repo_prefix = REPO_DIR + os.sep
sys.path[:] = [
    entry for entry in sys.path
    if os.path.abspath(entry) != REPO_DIR and not os.path.abspath(entry).startswith(repo_prefix)
]
sys.path.insert(0, REPO_DIR)

head = _run_git(["git", "-C", REPO_DIR, "rev-parse", "--abbrev-ref", "HEAD"])
if head != BRANCH:
    raise RuntimeError(
        f"Expected git branch {BRANCH} in {REPO_DIR}, found {head!r}."
    )
if not os.path.isfile(TUNING_SCRIPT):
    raise RuntimeError(
        f"Expected {TUNING_SCRIPT} on branch {BRANCH}, but the file is missing."
    )
if os.path.realpath(os.getcwd()) != os.path.realpath(REPO_DIR):
    raise RuntimeError(
        f"Working directory is {os.getcwd()!r}, expected {REPO_DIR}."
    )
print(f"Working directory: {os.getcwd()}")
print(f"Branch: {head}")
print(f"Tuning script: {TUNING_SCRIPT}")


In [ ]:
# Cell 2: Install project dependencies
!pip install -q -r requirements.txt
print("Dependencies installed successfully.")


In [ ]:
# Cell 3: Download Real Datasets & Generate SHA-256 Locked Splits
import os
import json
from src.data.download import load_real_datasets
from src.data.splits import build_records, create_stratified_splits, verify_no_leakage
from src.data.dedup import deduplicate_external_holdout

os.makedirs("data/splits", exist_ok=True)
os.makedirs("data/raw", exist_ok=True)

print("Ingesting Financial PhraseBank and SEntFiN 1.0 datasets...")
real_data = load_real_datasets(cache_dir="data/raw", execute=True)

configs = ["sentences_allagree", "sentences_75agree", "sentences_50agree"]
all_pb_texts = []
manifests = {}

for cfg in configs:
    cfg_data = real_data.get("financial_phrasebank", {}).get(cfg)
    if cfg_data:
        texts = cfg_data["texts"]
        labels = cfg_data["labels"]
        all_pb_texts.extend(texts)
        records = build_records(texts=texts, labels=labels, prefix=cfg, jaccard_threshold=0.80)
        manifest = create_stratified_splits(records=records, seed=42, ratios=(0.70, 0.15, 0.15), n_folds=5)
        manifest_path = f'data/splits/{cfg}_manifest.json'
        with open(manifest_path, "w", encoding="utf-8") as f:
            json.dump(manifest, f, indent=2)
        manifests[cfg] = manifest
        print(f"Created locked manifest for {cfg}: {len(records)} records -> {manifest_path}")

# External SEntFiN 1.0 holdout deduplication against all PhraseBank sentences
raw_sentfin = real_data.get("sentfin", [])
if isinstance(raw_sentfin, list) and len(raw_sentfin) > 0:
    filtered_holdout, stats = deduplicate_external_holdout(raw_sentfin, list(set(all_pb_texts)), threshold=0.80)
    if "sentfin_audit" in real_data:
        stats["upstream_audit"] = real_data["sentfin_audit"]
    with open("data/splits/sentfin_dedup.json", "w", encoding="utf-8") as f:
        json.dump({"stats": stats, "records": filtered_holdout}, f, indent=2)
    print(f"SEntFiN 1.0 Deduplication: Kept {stats['kept']} / {stats['total_input']} unambiguous unique sentences (removed {stats['removed_overlap']} near-duplicates)")


In [ ]:
# Cell 4: Comprehensive Data Leakage Audit
import glob
import json
import os
from src.data.splits import verify_no_leakage

print("=" * 70)
print("DATA LEAKAGE AUDIT VERIFICATION")
print("=" * 70)

manifest_files = sorted(glob.glob("data/splits/*_manifest.json"))
for mf in manifest_files:
    with open(mf, "r", encoding="utf-8") as f:
        m = json.load(f)
    audit = verify_no_leakage(m, jaccard_threshold=0.80)
    prefix = m.get("dataset_prefix", os.path.basename(mf))
    failed = (
        audit['id_overlap_count'] != 0
        or audit['hash_overlap_count'] != 0
        or audit['cluster_overlap_count'] != 0
        or audit['max_cross_split_jaccard'] >= 0.80
    )
    status = "FAILED" if failed else "PASSED"
    print(f"Manifest: {prefix}")
    print(f"  [+] ID Overlap: {audit['id_overlap_count']} ({status})")
    print(f"  [+] SHA-256 Overlap: {audit['hash_overlap_count']} ({status})")
    print(f"  [+] Cluster Overlap: {audit['cluster_overlap_count']} ({status})")
    print(f"  [+] Max Cross-Split Jaccard: {audit['max_cross_split_jaccard']:.4f} < 0.8000 ({status})")
    print(f"  [+] Partitions: Train={audit['split_sizes']['train']}, Val={audit['split_sizes']['val']}, Test={audit['split_sizes']['test']}")
    print()
    if failed:
        raise RuntimeError(f"Leakage audit failed for {prefix}")


In [ ]:
# Cell 5: Verify regenerated splits against data/splits/SHA256SUMS
# scripts/run_tuning.py calls the same check inside scripts/run_step2.py before any training.
import os
from scripts.run_step2 import verify_split_hashes

SPLITS_DIR = "data/splits"
SUMS_PATH = os.path.join(SPLITS_DIR, "SHA256SUMS")
if not os.path.isfile(SUMS_PATH):
    raise RuntimeError(f"Missing lock file: {SUMS_PATH}")

names = []
with open(SUMS_PATH, encoding="utf-8") as handle:
    for line in handle:
        line = line.strip()
        if not line:
            continue
        names.append(os.path.basename(line.split(None, 1)[1].strip().lstrip('*')))

paths = [os.path.join(SPLITS_DIR, name) for name in names]
missing = [path for path in paths if not os.path.isfile(path)]
if missing:
    raise RuntimeError("Missing locked split file(s): " + ", ".join(missing))

bad = verify_split_hashes(paths, SPLITS_DIR)
if bad:
    raise RuntimeError("SHA-256 mismatch against SHA256SUMS: " + ", ".join(bad))
for name in names:
    print(f"  [OK] {name}")
print("Locked splits match data/splits/SHA256SUMS.")


### Cached results on Google Drive
Mount Drive before training or before restoring a previous session. The backup directory is `/content/drive/MyDrive/essential_text_tuning`.

The next cell checks cached FinBERT runs against the tokenizer receipt, on Drive first and then on the local tree. A FinBERT directory is moved to `archive/invalid_finbert_tokenizer/<timestamp>/` when `result.json` says completed but `tokenizer_check.json` does not record vocabulary size 30,873 with a probe that is not entirely `[UNK]`. Completed `bert_base` runs stay in `selection/` and `final/`. The same check then runs before restore, so an archived FinBERT directory is not copied back into the active tree. `tokenizer_check.json` is part of the backup.

Resume uses a directory only when its `result.json` has status `completed`. FinBERT additionally requires a valid receipt. Partial `.tmp` directories are trained again and are not backed up.


In [ ]:
# Cell 6: Mount Google Drive, verify cached FinBERT tokenizer receipts, then restore completed runs
import json
import shutil
import subprocess
import sys
from pathlib import Path

DRIVE_BACKUP = None
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_BACKUP = Path("/content/drive/MyDrive/essential_text_tuning")
    DRIVE_BACKUP.mkdir(parents=True, exist_ok=True)
    print(f"Drive backup directory: {DRIVE_BACKUP}")
except ImportError:
    print("Not running in Colab. Completed runs stay in results/tuning on this machine.")

LOCAL_TUNING = Path("results/tuning")

def _is_completed_result(path: Path) -> bool:
    try:
        payload = json.loads(path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return False
    return payload.get("status") == "completed"

def _should_copy(path: Path, root: Path) -> bool:
    parts = path.relative_to(root).parts
    if any(part.endswith(".tmp") for part in parts):
        return False
    if path.name == "result.json":
        return _is_completed_result(path)
    if path.name == "predictions.npz":
        return _is_completed_result(path.with_name("result.json"))
    if path.name in {"tuning_results.csv", "final_results.csv", "selection.json"}:
        return True
    if path.name == "journal.jsonl":
        return True
    if path.name == "tokenizer_check.json":
        return _is_completed_result(path.with_name("result.json"))
    return False

def _needs_copy(src: Path, dest: Path) -> bool:
    if not dest.exists():
        return True
    src_stat, dest_stat = src.stat(), dest.stat()
    return src_stat.st_size != dest_stat.st_size or src_stat.st_mtime > dest_stat.st_mtime + 1

def _copy_tree(src_root: Path, dest_root: Path, *, keep_completed: bool) -> int:
    if src_root is None or not src_root.exists():
        return 0
    copied = 0
    for src in src_root.rglob("*"):
        if not src.is_file() or not _should_copy(src, src_root):
            continue
        dest = dest_root / src.relative_to(src_root)
        if keep_completed and dest.name == "result.json" and _is_completed_result(dest):
            continue
        if keep_completed and dest.name == "predictions.npz" and _is_completed_result(dest.with_name("result.json")):
            continue
        if not _needs_copy(src, dest):
            continue
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dest)
        copied += 1
    return copied

def count_completed(root: Path) -> int:
    if not root.exists():
        return 0
    total = 0
    for path in root.rglob("result.json"):
        if "archive" in path.relative_to(root).parts:
            continue
        if _is_completed_result(path):
            total += 1
    return total

def restore_tuning_results() -> int:
    """Copy completed Drive backups into results/tuning without replacing a completed local run."""
    if DRIVE_BACKUP is None:
        print("No Drive backup to restore.")
        return 0
    copied = _copy_tree(DRIVE_BACKUP, LOCAL_TUNING, keep_completed=True)
    print(f"Restored {copied} file(s) from Drive into {LOCAL_TUNING}.")
    print(f"Completed runs now on local disk: {count_completed(LOCAL_TUNING)}")
    return copied

def backup_tuning_results() -> int:
    """Copy completed local runs and written tables to Drive."""
    if DRIVE_BACKUP is None or not LOCAL_TUNING.exists():
        return 0
    return _copy_tree(LOCAL_TUNING, DRIVE_BACKUP, keep_completed=False)

def archive_invalid_finbert_tree(root) -> None:
    """Move FinBERT runs that fail the tokenizer receipt out of the active result tree."""
    if root is None:
        print("No tuning directory to archive.")
        return
    root = Path(root)
    if not root.exists():
        print(f"No tuning directory to archive: {root}")
        return
    completed = subprocess.run(
        [sys.executable, "scripts/run_tuning.py", "--archive-invalid-finbert", "--out-dir", str(root)],
        check=False,
    )
    if completed.returncode != 0:
        raise RuntimeError(f"Could not archive invalid FinBERT results in {root}.")

def recover_invalid_finbert() -> None:
    """Check the Drive backup before restore, then check the local tree."""
    archive_invalid_finbert_tree(DRIVE_BACKUP)
    archive_invalid_finbert_tree(LOCAL_TUNING)

recover_invalid_finbert()
restore_tuning_results()


In [ ]:
# Cell 7: Cache the two pinned checkpoints used by the tuning study
from huggingface_hub import snapshot_download
from scripts.run_step2 import FINETUNE_KEYS, MODEL_REVISIONS

for key in ("bert_base", "finbert_mlm"):
    name = FINETUNE_KEYS[key]
    revision = MODEL_REVISIONS[name]
    print(f"Caching {name} @ {revision}")
    snapshot_download(repo_id=name, revision=revision)
print("Pinned checkpoints are in the Hugging Face cache.")


In [ ]:
# Cell 8: Print the tuning plan. This does not train and does not write results/tuning.
# Until results/tuning contains every selection run, the plan says the final
# directories are not listed yet. That line is expected before training.
import subprocess
import sys

plan = subprocess.run(
    [sys.executable, "scripts/run_tuning.py", "--phase", "all"],
    check=False,
)
if plan.returncode != 0:
    raise SystemExit(plan.returncode)


### Training
The next cell verifies cached FinBERT receipts, restores the Drive backup, and then runs `scripts/run_tuning.py --phase all --execute`.

That command checks the FinBERT tokenizer before training: the vocabulary size must be 30,873, and profit, ebitda, and depreciation must not all be `[UNK]`. It then trains only directories that are not already completed. Completed BERT runs are left in place. Stage 1 writes the eight validation-only runs and selects one configuration per model. Stage 2 retrains that configuration with seeds 42, 100, and 2026 and scores both holdouts. The search remains learning rates `2e-5` and `3e-5` and batch sizes `16` and `32`.

Leave this runtime connected. Every 20 seconds the cell copies newly completed runs, including `tokenizer_check.json`, to Drive. When the cell finishes it copies the tables as well. After a disconnect, rerun from the first cell.


In [ ]:
# Cell 9: Run the tuning study and back up each completed run to Drive
import subprocess
import sys
import threading
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "This study needs a GPU runtime. Use Runtime → Change runtime type → GPU, then rerun from Cell 1."
    )
print(f"GPU: {torch.cuda.get_device_name(0)}")

recover_invalid_finbert()
restore_tuning_results()
stop = threading.Event()

def _watch() -> None:
    while not stop.is_set():
        copied = backup_tuning_results()
        if copied:
            print(f"Drive backup copied {copied} file(s).", flush=True)
        stop.wait(20)

thread = threading.Thread(target=_watch, name="drive-backup", daemon=True)
thread.start()
print("Drive backup watcher started.")
print("Launching scripts/run_tuning.py --phase all --execute")
completed = subprocess.run(
    [sys.executable, "scripts/run_tuning.py", "--phase", "all", "--execute"],
    check=False,
)
stop.set()
thread.join(timeout=5)
copied = backup_tuning_results()
print(f"Final Drive backup copied {copied} file(s).")
print(f"Completed runs on local disk: {count_completed(LOCAL_TUNING)}")
if completed.returncode != 0:
    raise SystemExit(completed.returncode)
print("Tuning study finished. Tables are in results/tuning and on Google Drive.")


In [ ]:
# Cell 10: Show the selection and final tables, then refresh the Drive copy
import csv
from pathlib import Path

def show_csv(path: Path) -> None:
    if not path.is_file():
        print(f"Not written yet: {path}")
        return
    with path.open(newline="", encoding="utf-8") as handle:
        rows = list(csv.DictReader(handle))
    print(f"{path} ({len(rows)} rows)")
    if not rows:
        return
    columns = list(rows[0].keys())
    print(" | ".join(columns))
    for row in rows:
        print(" | ".join(row[column] for column in columns))
    print()

show_csv(Path("results/tuning/tuning_results.csv"))
show_csv(Path("results/tuning/final_results.csv"))
selection = Path("results/tuning/selection.json")
if selection.is_file():
    print(f"Selection record: {selection}")
copied = backup_tuning_results()
print(f"Drive backup refreshed ({copied} file(s)).")


## Results Visualization and Analysis
This section reads completed results and writes figures. It does not train, and it does not call `scripts/run_tuning.py`.

It uses `results/tuning/tuning_results.csv`, `results/tuning/final_results.csv`, and `results/tuning/selection.json` when those files validate. If they are unavailable, it reads the same three files from `/content/drive/MyDrive/essential_text_tuning/`. Archived directories under `archive/` are not included.

The tables must contain 8 tuning configurations and 6 final runs, with seeds 42, 100, and 2026 for each model. The selected hyperparameters are checked against validation Macro-F1. Equal validation scores keep the lower learning rate, then the smaller batch size.

Figure 1 shows validation Macro-F1 for both models on one color scale, with the selected cell outlined. Figure 2 shows the three-seed mean Macro-F1 on the PhraseBank test set and on SEntFiN, with sample standard deviation. Figure 4 shows each seed separately. Figure 3 compares those tuned scores with the original fixed recipe (learning rate `2e-5`, batch size `16`) stored in `results/step2`. If those Step 2 files cannot be matched on dataset, metric, seed, and recipe, Figure 3 is omitted and the summary says why. No substitute values are inserted.

The axis for the bar charts starts at 0. Small differences are therefore not stretched. The summary states that seed-level differences are descriptive and are not a significance test.

Figures are written to `results/tuning/figures/` as 300 DPI PNG files and vector PDF files. `RESULTS_SUMMARY.md` in that directory includes the selected hyperparameters, the validation scores, the final means, and Markdown image links for the README.


In [ ]:
# Results visualization. This cell does not train.
from pathlib import Path

from IPython.display import Image, Markdown, display

from src.analysis.tuning_figures import generate_from_available_results

drive_root = Path("/content/drive/MyDrive/essential_text_tuning")
try:
    from google.colab import drive
except ImportError:
    drive_root = None
else:
    if not drive_root.parent.exists():
        drive.mount("/content/drive")

report = generate_from_available_results(
    local_root=Path("results/tuning"),
    figures_dir=Path("results/tuning/figures"),
    step2_runs=Path("results/step2/runs"),
    drive_root=drive_root,
)
if report.get("source_note"):
    print(report["source_note"])
if report.get("original_skip_reason"):
    print(report["original_skip_reason"])
print(f"Wrote {report['summary_path']}")
display(Markdown(report["summary_markdown"]))
for path in report["figures"]:
    if path.endswith(".png"):
        display(Image(filename=path))


In [ ]:
# Copy figures and RESULTS_SUMMARY.md to Google Drive. This cell does not train.
import shutil
from pathlib import Path

FIGURE_DIR = Path("results/tuning/figures")
REQUIRED = (
    "hyperparameter_heatmaps.png",
    "hyperparameter_heatmaps.pdf",
    "final_model_comparison.png",
    "final_model_comparison.pdf",
    "per_seed_performance.png",
    "per_seed_performance.pdf",
    "RESULTS_SUMMARY.md",
)
missing = [name for name in REQUIRED if not (FIGURE_DIR / name).is_file()]
if missing:
    raise RuntimeError(
        "Missing figure outputs: " + ", ".join(missing) + ". Run the visualization cell first."
    )

drive_figures = Path("/content/drive/MyDrive/essential_text_tuning/figures")
try:
    from google.colab import drive
except ImportError:
    print("Not running in Colab. Figures remain in results/tuning/figures.")
else:
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    drive_figures.mkdir(parents=True, exist_ok=True)
    for path in sorted(FIGURE_DIR.iterdir()):
        if path.suffix.lower() in {".png", ".pdf", ".md"}:
            shutil.copy2(path, drive_figures / path.name)
            print(f"Copied {path.name} to {drive_figures}")
